<h1 style="text-align: center;">HEISENBERG-BASED FAULT LOCALIZATION (HBFL)</h1>

<h2 style="text-align: center;">Download Initial Dependencies</h2>

In [1]:
import numpy as np
import pandas as pd
import json
import math
from tqdm import tqdm
from qiskit.quantum_info import SparsePauliOp, Operator, Pauli, Clifford
import qiskit.qasm2 as q2
from qiskit import QuantumCircuit
from IPython.display import display

from heisenbergUtilities import *

In [2]:
from qiskit_aer import AerSimulator
simulator = AerSimulator()

print(simulator.available_devices())

('CPU',)


<h2 style="text-align: center;">CONTROL PANEL</h2>

In [2]:
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#-------------------------------------------------------------------------GENERAL PARAMETERS---------------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
ORIGINAL_QASM = "realamprandom_5_qubits.qasm"                                              #| The original algorithm QASM circuit file name.                           #
ORIGINAL_QASM_FOLDER = "SBFL_circuits/MQTBench/"                                           #| The folder containing the original QASM circuit file.                    #
QASM_MUTANT_FOLDER = "SBFL_circuits/QMutBenchMutants/Mutants_realamprandom_5_qubits/"      #| The folder containing all mutants related to the original QASM circuit.  #
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#-------------------------------------------------------------------------SB-QOPS PARAMETERS---------------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
RUNS = 10                                                                       #| Number of times SB-QOPS will run to find a failing or passing test case.            #
SHOTS = 20000                                                                   #| Number of shots for SB-QOPS to use to create the compact program specification.     #
EPOCH = 80                                                                      #| Number of epochs SB-QOPS will run to navigate the search space of test cases.       #
SBQOPS_TOL = 0.1                                                                #| Tolerance value SB-QOPS uses to determine if a test case is failing or passing.     #
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
#---------------------------------------------------------------------HEISENBERG SBFL PARAMETERS-----------------------------------------------------------------------#
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#
ATOL = 1e-4                                                                     #| The tolerance value used to prune negligible magnitudes during Pauli propagation.   #
MAX_TERMS = 100                                                                 #| The maximum number of terms to keep during Pauli propagation. If None, use all.     #
SEARCH_STEP = 3                                                                 #| The search step size used during Pauli propagation. If None, pauli-prop uses 4.     #
VERBOSE = 0                                                                     #| A boolean value to determine if the program should print out detailed information.  #
#----------------------------------------------------------------------------------------------------------------------------------------------------------------------#


Generate the linked list of all operations that take place in the inverse circuit

In [3]:
"""
LinkedListNode: A class that holds information related to a gate in a quantum circuit. 

PROPERTIES:
    - value (String): The name of the quantum gate
    - depth (int): The depth of the gate in the circuit
    - count (Int): The probability that the gate will meaningfully change the Pauli string.
    - next (LinkedListNode): The next gate in the circuit

"""

class LinkedListNode:
    def __init__(self, name="Initial", depth=0, count=0,  next=None):
        self.value = name
        self.depth = depth
        self.next = next
        self.count = count

class LinkedList:
    def __init__(self):
        self.head = None

    def append(self, name, depth):
        new_node = LinkedListNode(name, depth)
        if not self.head:
            self.head = new_node
            return
        last_node = self.head
        while last_node.next:
            last_node = last_node.next
        last_node.next = new_node

    def mark(self, depth):
        current_node = self.head
        while current_node:
            if current_node.depth == depth:
                current_node.count += 1
                return
            current_node = current_node.next

    def reset(self):
        current_node = self.head
        while current_node:
            current_node.count = 0
            current_node = current_node.next

<h3>Download MQT Benchmark circuits.</h3>

We'll start with DJ, RealAmpRandom, and GHZ since SB-QOPS caught those mutants 100% of the time for 5 qubit circuits



<h3>Generate mutants.</h3>

We'll start with mutants that add an unnecessary gate, then add mutants that replace a certain gate later.

The mutants were downloaded from QMutBench, choosing specifically mutants that added a gate anywhere with 0-10% survival scores

<h2 style="text-align: center;"> SB-QOPS </h2>

This is where we will use SB-QOPS on a provided circuit and its mutants

For a single mutant, it took about 2 minutes to generate a 20 test suite (10 fail, 10 pass) on an RTX 4090 SUPER. It can only be run on a Linux environment since the AER Simulator does not support Windows

There are 113 mutants for the DJ algorithm in use: 226 minutes or 3 3/4 hours

There are 89 mutants for the GHZ algorithm in use: 178 minutes or about 3 hours

There are 125 mutants for the RealAmpRandom algorithm in use: 250 minutes or just over 4 hours

BUT this cell should only need to be run once per algorithm with mutants under test. After it saves to the csv file at the end, this cell can be commented out as to not accidentally run it again. 

In the future if we want to test this methodology on higher qubit circuits or other algorithms, we'll likely either want to reduce the number of tests in the suite or the number of mutants we're analyzing

In [ ]:
import QOPS as qops
from QOPS_test import get_compact_program_specification_Z
from pathlib import Path

correct_circuit = load_program(ORIGINAL_QASM, ORIGINAL_QASM_FOLDER).copy()
correct_list = LinkedList()
depth = 1

#Append each gate to the linked list
for instruction in correct_circuit.data:
    correct_list.append(instruction.name + " " + str(depth), depth)
    depth += 1

ga_result = pd.DataFrame(columns=['Program',"path_to_mutant",'catch_avg','avg_fitness','failing_testcases', 'passing_testcases'])
program_history = {}

#program_specification = The compact program specification. SB-QOPS needs this to generate failing test cases
program_specification = get_compact_program_specification_Z(correct_circuit, shots=SHOTS)

#mutants = A python list of qiskit QuantumCircuit variables with a mutation in them
mutants = []
mutants_names = []
root = Path(QASM_MUTANT_FOLDER)
for qasm_mutant in root.rglob("*.qasm"):
    mutants.append(load_program(qasm_mutant.name, QASM_MUTANT_FOLDER))
    mutants_names.append(qasm_mutant.name)

for mutant_index,mutant in enumerate(mutants):
    tester = qops.Circuit_Tester(CUT=mutant)
    tester.set_applicable_families_Z(program_specification)
    mutants_per_run = []
    fitness_per_run = []
    failing_testcases_per_run = []
    history_per_run = []

    print("NOW RUNNING TO FIND FAILING TESTS")
    #For a predefined number of attempts, try to find a set of failing test cases (output is above predefined tolerance)
    for runs in range(RUNS):
        killed = 0
        pauli = {}
        fitness = 0
        for i in range(len(tester.applicable_families)):
            best_function,testcase, history = tester.run_mealoneplusone(i, EPOCH)
            if best_function > SBQOPS_TOL:
                killed = 1
                pauli = testcase
                fitness = best_function
                break
        mutants_per_run.append(killed)
        failing_testcases_per_run.append(pauli)
        fitness_per_run.append(fitness)
        history_per_run.append(history)

    avg_score = np.mean(mutants_per_run)
    avg_fitness = np.mean(fitness_per_run)

    #Here is our new, modified algorithm from the SB-QOPS method that attempts to find passing test cases as well. We'll need them for SBFL
    passing_testcases_per_run = []

    print("NOW RUNNING TO FIND PASSING TESTS")
    for runs in range(RUNS):
        pauli = {}
        fitness = 0
        for i in range(len(tester.applicable_families)):
            best_function, testcase, history = tester.run_reverse_mealoneplusone(i,EPOCH)
            if best_function < SBQOPS_TOL:
                pauli = testcase
                break
        passing_testcases_per_run.append(pauli)

    #Replace static program file with "program_name" when ready to test fully
    """
    ga_result: A pandas dataframe with the following columns
        Program: Name of the mutant quantum circuit file
        Path_To_Mutant: Path to the mutant file
        Catch_Avg: The average number of times the mutant was identified by SB-QOPS
        Avg_Fitness: The average fitness of the search algorithm used. Higher usually indicates higher Catch_Avg
        Failing_Testcases: A list of dicts indicating the set of Pauli strings and the weights that should be applied to generate a failing test case
        Passing_Testcases: A list of dicts indicating the set of Pauli strings and the weights that should be applied to generate a passing test case
    """
    ga_result = pd.concat([pd.DataFrame([[mutants_names[mutant_index],QASM_MUTANT_FOLDER,avg_score,avg_fitness,json.dumps(failing_testcases_per_run), json.dumps(passing_testcases_per_run)]],columns=ga_result.columns),ga_result],ignore_index=True)
    ga_result.to_csv(f'dj_ga_result', index=False)

ga_result is a slightly altered pandas Dataframe with similar structure found in SB-QOPS's results folder. The differences are changing the column "Program" to be the name of the mutant file instead of the original quantum circuit, changing "Mutant" to be the path to the mutant file instead of being an arbitrary index value, and adding a passing_testcases column that returns Pauli strings and coefficients that generate passing tests.

Now we want to run SBFL using Heisenberg evolution trees on each circuit placed in ga_result

Evolution analysis tends to take about 5 minutes for 10 failing tests on a very complex algorithm such as QNN, so about 10 minutes for 20 total tests

DJ was a relatively small algorithm with few to no branching gates, so it only ended up taking about 5-6 minutes to evolve and analyze all 113 DJ mutants

About 890 minutes for GHZ, or 14.8 hours to evolve and analyze all 89 GHZ mutants

About 1250 minutes for RealAmpRandom, or 20.8 hours to evolve and analyze all 125 RealAmpRandom mutants

The runtime of this code segment is directly tied to the depth of the circuit being analyzed and the number of 2-branching gates such as rotation or phase gates that exist in the circuit.

This is something to note in the final paper. Regardless of accuracy this methodology takes a long time to run. If results are promising, then we'll want to look into the tradeoffs between EXAM and hyperparameters to speed this up. Primarily the atol, max_terms, and search_step parameters in the evolve_pauli_exact method in HeisenbergUtilities.py


<h2 style="text-align: center;">HEISENBERG EVOLUTIONS (PAULI PROPAGATION)</h2>

In [ ]:
from heisenbergTree import *

ga_result = pd.read_csv("dj_ga_result")
tarantula_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','exam_score'])
barinel_sbfl_result = pd.DataFrame(columns=['Program','path_to_mutant','exam_score'])

#For each mutant of a circuit, run the SBFL implementation
for mutant, path in zip(ga_result.loc[:,"Program"], ga_result.loc[:,"path_to_mutant"]):
    print("Now evolving the following mutant: ", mutant)

    #Extract the raw test cases found for each mutant
    desired_failing_testcases = ga_result.loc[(ga_result["Program"] == mutant), ["failing_testcases"]]
    desired_passing_testcases = ga_result.loc[(ga_result["Program"] == mutant), ["passing_testcases"]]
    raw_failing_testcases = desired_failing_testcases["failing_testcases"].values[0].split("}")
    raw_passing_testcases = desired_passing_testcases["passing_testcases"].values[0].split("}")

    #Remove empty tests
    raw_failing_testcases = remove_null_tests(raw_failing_testcases)
    raw_passing_testcases = remove_null_tests(raw_passing_testcases)

    circuit_inverse = load_program(mutant,path).copy().inverse()  # Invert to track backward evolution of the operator
    forward_mutant = load_program(mutant, path).copy()

    #Create the linked list of operations in the inverse circuit
    operation_list = LinkedList()
    depth = 0

    #Append each gate to the linked list
    operation_list.append("Initial", depth)
    for instruction in circuit_inverse.data:
        depth += 1
        operation_list.append(instruction.name + " " + str((circuit_inverse.size()-depth)), depth)

    forward_list = LinkedList()
    depth = 1
    for instruction in forward_mutant.data:
        forward_list.append(instruction.name + " " + str(depth), depth)
        depth += 1

    #Create list of nodes in linked list. Somewhere down the road remove the linked list implementation. It's unnecessary and giving me a headache
    node_list = []
    node = operation_list.head
    while node:
        node_list.append(node)
        node = node.next

    #Perform Pauli propagation (Heisenberg evolution) for all test cases. Returns a dataframe with all counts for all operations
    global_frame_fail = heisenberg_evolve(circuit_inverse, operation_list, raw_failing_testcases, pass_fail = "fail", atol = ATOL, max_terms = MAX_TERMS, search_step = SEARCH_STEP)
    global_frame_pass = heisenberg_evolve(circuit_inverse, operation_list, raw_passing_testcases, pass_fail = "pass", atol = ATOL, max_terms = MAX_TERMS, search_step = SEARCH_STEP)

    global_frame = pd.concat([global_frame_fail, global_frame_pass], ignore_index=True)

    global_frame = global_frame.drop(["Initial"],axis=1)

    tarantula_scores = tarantula(global_frame)
    barinel_scores = barinel(global_frame)

    if VERBOSE:
        print("BARINEL SCORES")
        display(barinel_scores)
        print("TARANTULA SCORES")
        display(tarantula_scores)
    
    # Here is where analysis of the methodology begins. 
    # We first extract the position of the erroneous gate by comparing it to the original MQT gate
    # NOTE: This method is intended for single-added gates for now. Other types of mutants will require other methods later
    error_gate_location = find_erroneous_gate(forward_list, correct_list)

    if VERBOSE:
        print("ERROR GATE LOCATION:")
        print(error_gate_location)

    # Calculate the EXAM score for Barinel by counting the number of gates we would have to observe before we reach the erroneous gate.
    gates_observed = 0
    barinel_exam_score = 0
    for col_name, col_date in barinel_scores.items():
        gates_observed += 1

        #Extract depth from column name
        gate_depth = int(col_name.split()[1])

        if gate_depth == error_gate_location:
            barinel_exam_score = gates_observed/(circuit_inverse.size()+1)
            break

    # Calculate the EXAM score for Tarantula by counting the number of gates we would have to observe before we reach the erroneous gate.
    gates_observed = 0
    tarantula_exam_score = 0
    for col_name, col_date in tarantula_scores.items():
        gates_observed += 1

        #Extract depth from column name
        gate_depth = int(col_name.split()[1])

        if gate_depth == error_gate_location:
            tarantula_exam_score = gates_observed/(circuit_inverse.size()+1)
            break

    # Here we store the results from the HBFL analysis for both barinel and tarantula into a csv file for later analysis.
    barinel_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,barinel_exam_score]],columns=barinel_sbfl_result.columns),barinel_sbfl_result],ignore_index=True)
    barinel_sbfl_result.sort_values(by=['exam_score'], ascending=False)
    barinel_sbfl_result.to_csv(f'dj_barinel_sbfl_result.csv', index=False)

    tarantula_sbfl_result = pd.concat([pd.DataFrame([[mutant,QASM_MUTANT_FOLDER,tarantula_exam_score]],columns=tarantula_sbfl_result.columns),tarantula_sbfl_result],ignore_index=True)
    tarantula_sbfl_result.sort_values(by=['exam_score'], ascending=False)
    tarantula_sbfl_result.to_csv(f'dj_tarantula_sbfl_result.csv', index=False)

if VERBOSE:
    display(barinel_sbfl_result)
    display(tarantula_sbfl_result)
    

Now evolving the following mutant:  AddGate_p_inGap_7_.qasm


100%|██████████| 10/10 [00:01<00:00,  8.98it/s]


,u 13,u 11,cx 10,p 9,h 6,cx 5,u 3,u 2,h 1,cx 12,h 8,cx 7,u 4,h 0
sum,0.614021,0.580587,0.580587,0.580587,0.580587,0.580587,0.580587,0.580587,0.580587,0.580587,0.580587,0.580587,0.580587,0.580587


h 1
h 2
u 3
u 4
u 5
cx 6
h 7
cx 8
h 9
p 10
10
Now evolving the following mutant:  AddGate_sx_inGap_1_.qasm


100%|██████████| 10/10 [00:01<00:00,  9.11it/s]


...

h 1
h 2
u 3
u 4
u 5
p 6
6
Now evolving the following mutant:  AddGate_ccx_inGap_4_.qasm


100%|██████████| 10/10 [00:02<00:00,  4.71it/s]


,u 13,cx 12,h 9,cx 8,h 7,ccx 6,cx 5,u 4,u 3,u 2,h 1,h 0,u 11,cx 10
sum,0.662976,0.612934,0.612934,0.612934,0.612934,0.612934,0.612934,0.612934,0.612934,0.612934,0.612934,0.612934,0.612934,0.612934


h 1
h 2
u 3
u 4
u 5
cx 6
ccx 7
7


,Program,path_to_mutant,exam_score
0,AddGate_ccx_inGap_4_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.6
1,AddGate_p_inGap_3_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.533333
2,AddGate_s_inGap_4_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.333333
3,AddGate_h_inGap_6_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.133333
4,AddGate_h_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.2
...,...,...,...
108,AddGate_x_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.6
109,AddGate_cx_inGap_3_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.466667
110,AddGate_y_inGap_9_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.666667
111,AddGate_sx_inGap_1_.qasm,SBFL_circuits/QMutBenchMutants/Mutants_dj_5_qu...,0.8
